# Calculating Betas for the Market's Stocks #

### Calculating the Betas for all the Stocks in the Universe ###

In [1]:
# Import Libraries

# Data Management
import pandas as pd
import numpy as np

# Plots
import matplotlib.pyplot as plt

# Statistics
from scipy.stats import norm
import statsmodels.api as sm

# Handle Files
import sys
import os

# Import Local Functions
sys.path.append(os.path.abspath("../source"))
from other_data_functions import wexp
from regression_toolkit import add_constant
from regression_toolkit import rolling_wls_regression
from factors_toolkit import compute_factor_contributions
from factors_toolkit import compute_residual_returns

In [2]:
# Get the important data for the Risk-Free Rate
rfr = pd.read_csv(r"../additional_data/risk_free_rate.csv")
rfr.set_index('Date', inplace=True)
rfr.index = pd.to_datetime(rfr.index)

# Get the important data for the S&P500
benchmark = pd.read_csv(r'../additional_data/benchmark_returns.csv')
benchmark.set_index('Date', inplace=True)
benchmark.index = pd.to_datetime(benchmark.index)

In [3]:
# Import Data
returns = pd.read_csv(r'../additional_data/stocks_returns.csv')
returns = returns.rename(columns={'Unnamed: 0':'Date'})
returns.set_index('Date', inplace=True)
returns.index = pd.to_datetime(returns.index)

returns

In [4]:
# Define the tickers
tickers = returns.columns

In [5]:
# Set the arrays
y_matrix = returns.subtract(rfr['risk_free_rate'], axis=0)
x_matrix = benchmark['benchmark_returns'] - rfr['risk_free_rate']
x_matrix.name = 'market_premium'

# Add constant
x_matrix = add_constant(x_matrix)

x_matrix

In [6]:
# Let us get the betas of each stock
coefficients = rolling_wls_regression(
    y_matrix,
    x_matrix
)

coefficients

In [7]:
# Create Alpha's DataFrame
alpha_df = coefficients['alphas']

alpha_df

In [8]:
# Create Beta's DataFrame
betas_df = coefficients['betas']

betas_df

In [9]:
# Create the Sigma's DataFrame
sigma_df = coefficients['sigmas']

sigma_df

In [10]:
# Save the betas
#alpha_df.to_csv(r"../additional_data/capm_ralpha.csv")
#betas_df.to_csv(r"../additional_data/capm_rbetas.csv")
#sigma_df.to_csv(r"../additional_data/capm_rsigma.csv")

In [11]:
# We can use StatsModels efficiently to get the betas for the whole history
betas_list = []

# Loop to Obtain Betas and Alpha + Residuals
for ticker in tickers:
    # Define series
    y_series = y_matrix[ticker].dropna()
    
    # Set the Window
    window = len(y_series)
    weights = window * wexp(window, window/2)
    
    # Define weights
    model = sm.WLS(y_series, x_matrix.loc[y_series.index], weights=weights)
    results = model.fit()
    
    beta = results.params.iloc[1]
    
    betas_list.append(beta)

# Create Beta Series
betas_series = pd.Series(betas_list, index=tickers)
betas_series.name = 'history_beta'

betas_series

In [12]:
# Plot
ticker = 'AAPL'

# Mean
mean = betas_df[ticker].mean()

# Create the Plot
plt.figure(figsize=(10, 6))
plt.plot(betas_df[ticker], label=f'{ticker} Beta', color='blue', alpha=0.7)
plt.axhline(y=betas_series.loc[ticker], color='red', linestyle='dashed', label=f'{ticker} Historical Beta')
plt.axhline(y=mean, color='black', linestyle='dashed', label=f'{ticker} Mean Beta')

# Config
plt.title('Beta Time Series')
plt.xlabel('Time')
plt.ylabel('Betas')
plt.legend()

# Show
plt.grid()
plt.show()

### Beta Analytics ###

In [13]:
# Create the Plot
plt.figure(figsize=(10, 6))
plt.plot(betas_df.mean(axis=1), label='Betas Mean', color='orange', alpha=0.7)
plt.axhline(y=1, color='black', linestyle='dashed')

# Config
plt.title('Beta Time Series')
plt.xlabel('Time')
plt.ylabel('Betas')
plt.legend()

# Show
plt.grid()
plt.show()

In [24]:
# Calculate Mean and Standard Deviation
mu = betas_series.mean()
sigma = betas_series.std()

# Create Histogram
plt.figure(figsize=(10, 6))
plt.hist(betas_series, bins=30, density=True, color='lightskyblue', alpha=0.5, edgecolor='black', label='Betas Distribution')

# Generate the Values of the Normal Distribution
x = np.linspace(betas_series.min(), betas_series.max(), 100)
y = norm.pdf(x, mu, sigma)

# Graph the Real Normal Distribution
plt.plot(x, y, color='black', linestyle='solid', linewidth=2, label='Normal Distribution')

# Reference Lines
plt.axvline(x=mu, color='black', linestyle='dashed', label='Mean Returns')
plt.axvline(x=betas_series.median(), color='red', linestyle='dashed', label='Median Returns')
plt.axvline(x=mu + sigma, color='grey', linestyle='dashed')
plt.axvline(x=mu - sigma, color='grey', linestyle='dashed')

# Config
plt.title('Betas Histogram with Normal Distribution')
plt.xlabel('Return')
plt.ylabel('Density')

# Legends and Grid
plt.legend()
plt.grid(True)

# Save
plt.savefig(r"..\plots\betas_histogram.jpg", dpi=300, bbox_inches="tight")

# Show
plt.show()

### Comparing Residuals ###

In [15]:
# Compute Residuals in the Alternative Way
stock = 'JPM'

# Cut DataFrames
stock_beta = betas_df[stock].dropna()
r_i = y_matrix[stock].loc[stock_beta.index]
r_m = x_matrix['market_premium'].loc[stock_beta.index]

In [16]:
# Calculate factor returns
r_f = compute_factor_contributions(r_m, stock_beta)

# Calculate residual returns
residual_returns = compute_residual_returns(r_i, r_m, stock_beta)
residual_returns.name = 'residual_returns'

residual_returns

In [17]:
# Create the Plot
plt.figure(figsize=(10, 6))
plt.plot(returns[stock].loc['2000':].cumsum(), label=f'{stock} Returns', alpha=0.7)
#plt.plot(benchmark.loc['2000':].cumsum(), label='Benchmark Returns', alpha=0.7)
plt.plot(r_i.cumsum(), label=f'{stock} Excess Returns', alpha=0.7)
plt.plot(r_f.cumsum(), label=f'{stock} Factor Returns', alpha=0.7)
plt.plot(residual_returns.cumsum(), label=f'{stock} Residual Returns', alpha=0.7)
plt.axhline(y=0, color='black', linestyle='dashed')

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns')
plt.legend()

# Show
plt.grid()
plt.show()

The method using the rolling betas adds so much noise to the calculation of the residual returns, especially because sometimes the alpha coefficient has a bias. So we prefer to use the residuals of the whole time stamp regression.

In [18]:
# Calculate the Residual Returns for every stock
residual_returns_dict = {}

# Create the Loop to Obtain the Betas
for ticker in tickers:
    
    index = betas_df[ticker].dropna().index
    
    df = compute_residual_returns(
        y_matrix[ticker].loc[index], 
        x_matrix['market_premium'].loc[index],
        betas_df[ticker].loc[index]
    )
    
    residual_returns_dict[ticker] = df

# Create the DataFrame
df_residual_returns = pd.DataFrame.from_dict(residual_returns_dict)

In [19]:
df_residual_returns

In [20]:
# Save the data
betas_series.to_csv(r"../additional_data/capm_hbetas.csv")
df_residual_returns.to_csv(r"../additional_data/capm_residual_returns.csv")